In [1]:
!pip install langchain langchain-groq python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. Setup

In [ ]:
import os

from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model=os.getenv("GROQ_MODEL", "openai/gpt-oss-120b"),
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3,
    max_retries=5,
)

llm.invoke("Reply with just: ok").content

'ok'

### 2. Prompt and test conversation

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant. Answer in 1-3 short sentences."),
        MessagesPlaceholder("history"),
        ("human", "{input}"),
    ]
)

# early turns plant facts, the last two turns check if the agent still has them
conversation = [
    "Hi, my name is Sudev and I'm a final year computer engineering student.",
    "I'm building a study planner app for my college project.",
    "I prefer Python for the backend and I want to keep it simple.",
    "What's a good database for a small project like this?",
    "Any tip for structuring the API?",
    "What's my name and what am I building?",
    "Given everything you know about me, which backend framework should I use?",
]

### 3. Agent without memory

In [ ]:
chat_chain = chat_prompt | llm | StrOutputParser()


def chat_no_memory(text: str) -> str:
    # every turn starts from an empty history
    return chat_chain.invoke({"input": text, "history": []})

### 4. Short-term memory (sliding window over conversation history)

In [5]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import AIMessage, HumanMessage, trim_messages

WINDOW_SIZE = 4  # last 4 messages = last 2 exchanges

window_trimmer = trim_messages(
    max_tokens=WINDOW_SIZE,
    token_counter=len,  # count messages, not tokens
    strategy="last",
    start_on="human",
)

window_history = InMemoryChatMessageHistory()


def chat_window_memory(text: str) -> str:
    # full history is stored, but the model only sees the last few messages
    context = window_trimmer.invoke(window_history.messages)
    reply = chat_chain.invoke({"input": text, "history": context})
    window_history.add_messages([HumanMessage(text), AIMessage(reply)])
    return reply

### 5. Summary-based memory (running summary + recent messages)

In [6]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, SystemMessage, get_buffer_string

summary_prompt = ChatPromptTemplate.from_template(
    """Progressively summarize the conversation, adding onto the previous summary.
Keep every fact about the user (name, goals, preferences) and drop small talk.
Return only the new summary.

Current summary:
{summary}

New lines of conversation:
{new_lines}"""
)

summary_chain = summary_prompt | llm | StrOutputParser()


class SummaryChatHistory(BaseChatMessageHistory):
    """Keeps the last few messages as-is and folds older ones into a summary."""

    def __init__(self, max_recent: int = 4):
        self.max_recent = max_recent
        self.summary = ""
        self.recent: list[BaseMessage] = []

    @property
    def messages(self) -> list[BaseMessage]:
        if not self.summary:
            return list(self.recent)
        memo = SystemMessage(f"Summary of the earlier conversation:\n{self.summary}")
        return [memo, *self.recent]

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.recent.extend(messages)
        if len(self.recent) > self.max_recent:
            old = self.recent[: -self.max_recent]
            self.recent = self.recent[-self.max_recent :]
            self.summary = summary_chain.invoke(
                {"summary": self.summary or "None", "new_lines": get_buffer_string(old)}
            )

    def clear(self) -> None:
        self.summary = ""
        self.recent = []


summary_history = SummaryChatHistory(max_recent=WINDOW_SIZE)


def chat_summary_memory(text: str) -> str:
    reply = chat_chain.invoke({"input": text, "history": summary_history.messages})
    summary_history.add_messages([HumanMessage(text), AIMessage(reply)])
    return reply

### 6. Run the same conversation through all three agents

In [7]:
agents = {
    "no_memory": chat_no_memory,
    "window_memory": chat_window_memory,
    "summary_memory": chat_summary_memory,
}

replies = {name: [] for name in agents}

for turn, text in enumerate(conversation, start=1):
    print(f"[{turn}] USER: {text}")
    for name, chat in agents.items():
        reply = chat(text)
        replies[name].append(reply)
        print(f"    {name:<15} {reply}")
    print()

[1] USER: Hi, my name is Sudev and I'm a final year computer engineering student.


    no_memory       Nice to meet you, Sudev! Good luck with your final year—let me know if you need any help with your projects or studies.


    window_memory   Nice to meet you, Sudev! Good luck with your final year—let me know if you need any help with your projects or studies.


    summary_memory  Nice to meet you, Sudev! How can I assist you today?

[2] USER: I'm building a study planner app for my college project.


    no_memory       That sounds like a great project! Let me know which part you need help with—design, scheduling algorithms, notifications, or something else.


...

    window_memory   I don’t have your name from our conversation, but based on what you’ve described you’re building a small planner‑style API (e.g., courses, tasks, deadlines) backed by a lightweight database such as SQLite.


    summary_memory  Your name is **Sudev**, and you’re building a **study‑planner app** for your college project.

[7] USER: Given everything you know about me, which backend framework should I use?


    no_memory       I don’t have enough details about your project’s language preferences, scale, or requirements to recommend a specific backend framework. Could you share what language you’re comfortable with, the type of application you’re building, and any performance or ecosystem considerations? That’ll help me suggest the best fit.


    window_memory   FastAPI is a great fit—it’s lightweight, async‑ready, gives automatic OpenAPI docs, and integrates cleanly with Pydantic models for your course/task schemas. It also scales easily if you later add authentication or more complex logic. If you prefer an even simpler setup, Flask works too, but FastAPI will speed development for a modern API.


    summary_memory  FastAPI is the better fit—it gives you automatic OpenAPI docs, built‑in request validation with Pydantic, and async support while still being easy to start with SQLite/SQLAlchemy. If you prefer an ultra‑minimal setup, Flask works too, but FastAPI will speed development and testing for your study‑planner app.



### 7. Compare the recall questions

In [8]:
recall_turns = range(len(conversation) - 2, len(conversation))

for i in recall_turns:
    print("Q:", conversation[i])
    for name in agents:
        print(f"  {name:<15} {replies[name][i]}")
    print()

Q: What's my name and what am I building?
  no_memory       I don’t have any information about your name or what you’re building. If you share those details, I’ll be happy to help!
  window_memory   I don’t have your name from our conversation, but based on what you’ve described you’re building a small planner‑style API (e.g., courses, tasks, deadlines) backed by a lightweight database such as SQLite.
  summary_memory  Your name is **Sudev**, and you’re building a **study‑planner app** for your college project.

Q: Given everything you know about me, which backend framework should I use?
  no_memory       I don’t have enough details about your project’s language preferences, scale, or requirements to recommend a specific backend framework. Could you share what language you’re comfortable with, the type of application you’re building, and any performance or ecosystem considerations? That’ll help me suggest the best fit.
  window_memory   FastAPI is a great fit—it’s lightweight, async‑re

### 8. What each memory holds after the conversation

In [9]:
print("window memory: stored", len(window_history.messages), "messages, model sees", end=" ")
print(len(window_trimmer.invoke(window_history.messages)))

print("summary memory: model sees", len(summary_history.messages), "messages\n")
print("running summary:")
print(summary_history.summary)

window memory: stored 14 messages, model sees 4
summary memory: model sees 5 messages

running summary:
User Sudev is a final‑year computer engineering student building a study‑planner app for a college project. They prefer a simple Python backend, recommending Flask or FastAPI with SQLite via SQLAlchemy and basic CRUD endpoints; SQLite was advised as the ideal choice for a small, single‑user app, with PostgreSQL mentioned as a future option for multi‑user or remote access. For API structure, Sudev should group related routes using a Flask Blueprint or FastAPI APIRouter (e.g., `/tasks` with `GET /`, `POST /`, `PUT /{id}`, `DELETE /{id}`), keep handlers thin by validating input and delegating to a service layer that contains business logic and interacts with SQLAlchemy models, use Pydantic (FastAPI) or Marshmallow (Flask) schemas for request/response validation and serialization, and return consistent JSON with appropriate status codes (200/201 for success, 400 for bad input, 404 for mi

### 9. Observations

- **No memory:** each turn is answered in isolation, so it can't say the user's name or project and gives generic framework advice.
- **Short-term (window) memory:** the model only sees the last few exchanges. It handles follow-ups well, but facts from the first turns (name, project) have slid out of the window and are lost.
- **Summary memory:** older turns are compressed into a running summary while recent turns stay verbatim. The prompt stays small but the key facts survive, so the agent recalls the name and project and tailors its advice (simple, Python backend).
- Tradeoff: summary memory costs an extra LLM call whenever the buffer overflows and the summary can drop details, while a full buffer keeps everything but grows the prompt every turn.